# PII Masking 400k Results Analysis

This notebook analyzes `ner`/`langextract` alignment results and extracts all labels encountered in:

- `train_candidates.jsonl`
- `samples.jsonl`

It also identifies extra labels present in full `samples` that are missing from `train_candidates`.


In [ ]:
from pathlib import Path
import json
from collections import Counter

import pandas as pd

BASE_DIR = Path('../../../resources/data/pii-masking-400k/qwen3:8b-260502_1504')
TRAIN_PATH = BASE_DIR / 'train_candidates.jsonl'
SAMPLES_PATH = BASE_DIR / 'samples.jsonl'

for p in [TRAIN_PATH, SAMPLES_PATH]:
    print(p, 'exists=', p.exists())


In [ ]:
ENTITY_FIELDS = ['ner_predictions', 'langextract_predictions', 'final_entities']


def iter_jsonl(path):
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line:
                yield json.loads(line)


def collect_labels(path, fields=ENTITY_FIELDS):
    all_counter = Counter()
    per_field = {field: Counter() for field in fields}
    row_count = 0

    for obj in iter_jsonl(path):
        row_count += 1
        for field in fields:
            for ent in obj.get(field, []) or []:
                label = ent.get('label')
                if label:
                    all_counter[label] += 1
                    per_field[field][label] += 1

    return {
        'rows': row_count,
        'all_counter': all_counter,
        'per_field': per_field,
        'labels': set(all_counter.keys()),
    }


In [ ]:
train_stats = collect_labels(TRAIN_PATH)
samples_stats = collect_labels(SAMPLES_PATH)

print('train rows:', train_stats['rows'])
print('samples rows:', samples_stats['rows'])
print('train unique labels:', len(train_stats['labels']))
print('samples unique labels:', len(samples_stats['labels']))


In [ ]:
train_labels = train_stats['labels']
samples_labels = samples_stats['labels']

extra_in_samples = sorted(samples_labels - train_labels)

print('Labels in train_candidates:')
print(sorted(train_labels))

print('Extra labels found in samples (candidate pool):')
print(extra_in_samples)

In [ ]:
def counter_to_df(counter, name='count'):
    if not counter:
        return pd.DataFrame(columns=['label', name])
    df = pd.DataFrame(counter.items(), columns=['label', name])
    return df.sort_values(name, ascending=False).reset_index(drop=True)

train_df = counter_to_df(train_stats['all_counter'], 'train_count')
samples_df = counter_to_df(samples_stats['all_counter'], 'samples_count')

summary = (
    train_df
    .merge(samples_df, on='label', how='outer')
    .fillna(0)
)
summary['train_count'] = summary['train_count'].astype(int)
summary['samples_count'] = summary['samples_count'].astype(int)
summary['is_extra_in_samples'] = summary['train_count'] == 0
summary = summary.sort_values(['is_extra_in_samples', 'samples_count'], ascending=[False, False]).reset_index(drop=True)

summary


In [ ]:
# Inspect where extra labels come from (ner vs langextract vs final_entities)
extra_labels = set(extra_in_samples)

rows = []
for field, counter in samples_stats['per_field'].items():
    for label in sorted(extra_labels):
        rows.append({
            'field': field,
            'label': label,
            'count': counter.get(label, 0),
        })

extra_breakdown = pd.DataFrame(rows).sort_values(['label', 'count'], ascending=[True, False]).reset_index(drop=True)
extra_breakdown


In [ ]:
from pathlib import Path

# Optional: export label reports for downstream use
reports_dir = Path(BASE_DIR) / 'reports'
reports_dir.mkdir(parents=True, exist_ok=True)

(summary.sort_values('samples_count', ascending=False)
 .to_csv(reports_dir / 'label_counts_train_vs_samples.csv', index=False))

pd.DataFrame({'train_labels': sorted(train_labels)}).to_csv(
    reports_dir / 'train_candidate_labels.csv', index=False
)

pd.DataFrame({'extra_labels_from_samples': extra_in_samples}).to_csv(
    reports_dir / 'extra_labels_from_samples.csv', index=False
)

extra_breakdown.to_csv(reports_dir / 'extra_labels_field_breakdown.csv', index=False)

print('Saved reports to', reports_dir)


## Label Histograms


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

def plot_label_hist(counter, title, top_n=30):
    df = counter_to_df(counter, 'count').head(top_n)
    plt.figure(figsize=(12, 6))
    sns.barplot(data=df, x='label', y='count', palette='viridis')
    plt.title(title)
    plt.xlabel('Label')
    plt.ylabel('Count')
    plt.xticks(rotation=60, ha='right')
    plt.tight_layout()

plot_label_hist(train_stats['all_counter'], 'Train Candidates Label Distribution')
plot_label_hist(samples_stats['all_counter'], 'Samples Label Distribution')


## BIO Token-Level DataFrames (NER vs LangExtract)


In [ ]:
import re

TOKEN_PATTERN = re.compile(r'\S+')


def find_label_for_token(token_start, token_end, entities):
    best_label = 'O'
    best_overlap = 0
    for ent in entities or []:
        start = ent.get('start_char')
        end = ent.get('end_char')
        label = ent.get('label')
        if start is None or end is None or not label:
            continue
        if start == token_start and end == token_end:
            return label
        overlap = max(0, min(token_end, end) - max(token_start, start))
        if overlap > best_overlap:
            best_overlap = overlap
            best_label = label
    return best_label


def to_bio_df(path, sample_limit=None):
    rows = []
    sample_count = 0

    for obj in iter_jsonl(path):
        sample_count += 1
        if sample_limit is not None and sample_count > sample_limit:
            break

        text = obj.get('text', '') or ''
        ner_entities = obj.get('ner_predictions', []) or []
        lx_entities = obj.get('langextract_predictions', []) or []
        paragraph_id = obj.get('paragraph_id')

        for m in TOKEN_PATTERN.finditer(text):
            token = m.group(0)
            token_start, token_end = m.start(), m.end()
            ner_label = find_label_for_token(token_start, token_end, ner_entities)
            lx_label = find_label_for_token(token_start, token_end, lx_entities)

            rows.append({
                'paragraph_id': paragraph_id,
                'token': token,
                'ner_pred': ner_label,
                'lx_pred': lx_label,
                'match': ner_label == lx_label,
            })

    return pd.DataFrame(rows, columns=['paragraph_id', 'token', 'ner_pred', 'lx_pred', 'match'])


def export_bio_txt(df, out_path, tag_col='ner_pred', token_col='token', paragraph_col='paragraph_id'):
    out_path = Path(out_path)
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with out_path.open('w', encoding='utf-8') as f:
        for _, group in df.groupby(paragraph_col, dropna=False, sort=False):
            prev_label = 'O'
            for _, row in group.iterrows():
                token = str(row[token_col])
                label = str(row[tag_col]) if pd.notna(row[tag_col]) else 'O'

                if label == 'O':
                    bio_tag = 'O'
                elif label == prev_label:
                    bio_tag = f'I-{label}'
                else:
                    bio_tag = f'B-{label}'

                f.write(f"{token} {bio_tag}\n")
                prev_label = label
            f.write('\n')

    print('Saved BIO txt:', out_path)


In [ ]:
# Set sample_limit=None to process full files.
sample_limit = None

train_bio_df = to_bio_df(TRAIN_PATH, sample_limit=sample_limit)
samples_bio_df = to_bio_df(SAMPLES_PATH, sample_limit=sample_limit)

print('train_bio_df shape:', train_bio_df.shape)
print('samples_bio_df shape:', samples_bio_df.shape)


In [ ]:
train_bio_df[:10]


In [ ]:
samples_bio_df[:10]

In [ ]:
print('Train token-level match rate:', round(train_bio_df['match'].mean() * 100, 2), '%')
print('Samples token-level match rate:', round(samples_bio_df['match'].mean() * 100, 2), '%')


In [ ]:
from pathlib import Path

OUTPUT_DIR = '../../../resources/outputs/pii-masking-400k/reports'  # string or Path
reports_dir = Path(OUTPUT_DIR)
reports_dir.mkdir(parents=True, exist_ok=True)

# CSV exports
train_bio_out_csv = reports_dir / 'train_candidates_bio_tokens.csv'
samples_bio_out_csv = reports_dir / 'samples_bio_tokens.csv'
train_bio_df.to_csv(train_bio_out_csv, index=False)
samples_bio_df.to_csv(samples_bio_out_csv, index=False)
print('Saved:', train_bio_out_csv)
print('Saved:', samples_bio_out_csv)

# BIO txt exports (choose ner_pred or lx_pred)
train_bio_out_txt = reports_dir / 'train_candidates_bio.txt'


export_bio_txt(train_bio_df, train_bio_out_txt, tag_col='ner_pred')